In [1]:
import json
from sklearn.model_selection import train_test_split
import re
from tqdm import tqdm
from datasets import Dataset
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import shap
import numpy as np
import pickle

In [2]:
import logging
logging.getLogger('shap').setLevel(logging.WARNING) # turns off the "shap INFO" logs
logging.getLogger('matplotlib').setLevel(logging.WARNING) # turns off the progress bar

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
with open("/home/imruhi/Documents/RiskPerceptionSeafare/params.json", 'r') as f:
    PARAMS = json.load(f)

In [5]:
def clean_text(text):
    # remove tags from xml
    text = re.sub(r"<.*>", ' ', text)
    # remove indication of beginning of paragraph
    text = re.sub(r"^§ [\.\w]*\s*", ' ', text)
    # remove anything in [] 
    text = re.sub(r"\[.*\]", '', text)
    return text.lstrip().rstrip()


In [6]:
def load_dataset():
    dataset_path = f'{PARAMS["roberta_data_path"]}_{PARAMS["word_window"]}_filtered'
    all_excerpts = Dataset.load_from_disk(dataset_path).to_pandas()
    dataset = pd.DataFrame({"text":all_excerpts["text"], "label":all_excerpts["label"]}).dropna().drop_duplicates()
    print("Cleaning text")
    dataset["text"] = [clean_text(x) for x in tqdm(dataset["text"])]

    return dataset

In [7]:
def split_dataset(dataset, labels, train_size, val_size):
    label2id = {label: i for i, label in enumerate(labels)}
    id2label = {i: label for i, label in enumerate(labels)}
    print(label2id)  
    print(id2label)
    train_data, test_data = train_test_split(dataset, train_size=train_size, stratify=dataset["label"], random_state=42)
    test_data, val_data = train_test_split(test_data, train_size=val_size, stratify=test_data["label"], random_state=42)
    train_data["label"] = [label2id[x] for x in train_data["label"]]
    test_data["label"] = [label2id[x] for x in test_data["label"]]
    val_data["label"] = [label2id[x] for x in val_data["label"]]
    train_data = Dataset.from_pandas(train_data).remove_columns(["__index_level_0__"])
    test_data = Dataset.from_pandas(test_data).remove_columns(["__index_level_0__"])
    val_data = Dataset.from_pandas(val_data).remove_columns(["__index_level_0__"])
    
    return label2id, id2label, train_data, test_data, val_data

In [8]:
model_id = PARAMS["classi_finetune_model"]
dataset = load_dataset()
print(f"Size of dataset: {len(dataset)}")
labels = list(dataset["label"].unique())

label2id, id2label, train_data, test_data, val_data = split_dataset(dataset, labels, train_size=PARAMS["train_split"], val_size=PARAMS["val_split"])

print(f'Train: {Counter(train_data["label"])}')
print(f'Test: {Counter(test_data["label"])}')
print(f'Val: {Counter(val_data["label"])}')

Cleaning text


100%|██████████| 5859/5859 [00:00<00:00, 179285.08it/s]

Size of dataset: 5859
{'HIGH': 0, 'MEDIUM': 1, 'LOW': 2}
{0: 'HIGH', 1: 'MEDIUM', 2: 'LOW'}
Train: Counter({1: 2089, 2: 1662, 0: 936})
Test: Counter({1: 261, 2: 208, 0: 117})
Val: Counter({1: 261, 2: 208, 0: 117})


In [9]:
dataset["label"].value_counts()

label
MEDIUM    2611
LOW       2078
HIGH      1170
Name: count, dtype: int64

In [10]:
val_data.to_pandas().to_csv("val_data.csv")

In [11]:
# Load your fine-tuned BERT model and tokenizer
model_path = f'{PARAMS["save_model"]}{model_id.split("/")[-1]}_finetuned_16_12'
model_base = PARAMS["classi_finetune_model"]
tokenizer = AutoTokenizer.from_pretrained(model_base)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model = model.to("cuda")
model.eval()

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(256000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
        (a

In [12]:
preds = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
)


In [13]:
explainer = shap.Explainer(
    preds, seed=42
)

In [14]:
texts = val_data["text"]

In [ ]:
size = 10
shap_values = []
for itr in range(0, len(texts), size):
    print(f"Processing batch {itr} - {itr+size}")
    batch_texts = texts[itr:itr+size]
    batch_shap_values = explainer(batch_texts)
    with open(f"shap_batch_{itr}.pkl", "wb") as f:
        pickle.dump(batch_shap_values, f)
    shap_values.append(batch_shap_values)

In [16]:
all_class_shap = None
for i in range(0, len(texts), size):
    with open(f"shap_batch_{i}.pkl", "rb") as f:
        batch_sv = pickle.load(f)

    if all_class_shap is None:
        all_class_shap = batch_sv.values
    else:
        for c in range(len(batch_sv)):
            all_class_shap[c] = np.concatenate(
                [all_class_shap[c], batch_sv.values[c]],
                axis=0
            )

In [17]:
all_data = []

for itr in range(0, len(texts), size):
    batch_texts = texts[itr:itr+size]
    for x in batch_texts:
        all_data.append(tokenizer.tokenize(x))


In [18]:
len(all_data)

586

In [19]:
import numpy as np

def merge_tokens(tokens, values):
    merged_tokens = []
    merged_values = []

    current_token = ""
    current_value = None

    for t, v in zip(tokens, values):

        # RoBERTa word start token
        if t.startswith("▁"):

            # flush previous token
            if current_token != "":
                merged_tokens.append(current_token)
                merged_values.append(current_value)

            current_token = t[1:]  # remove Ġ
            current_value = v.copy()

        else:
            # continuation of same word
            current_token += t
            current_value += v

    # flush last token
    if current_token:
        merged_tokens.append(current_token)
        merged_values.append(current_value)

    return merged_tokens, np.array(merged_values)

In [20]:
from collections import defaultdict
import numpy as np

class_contribs = [defaultdict(list) for _ in range(3)]

for d, sv in zip(all_data, all_class_shap):

    merged_tokens, merged_values = merge_tokens(d, sv)

    for i, token in enumerate(merged_tokens):
        for c in range(3):
            class_contribs[c][token].append(merged_values[i, c])

In [21]:
agg = []

for c in range(3):
    token_scores = {}
    for token, vals in class_contribs[c].items():
        token_scores[token] = np.mean(vals)  # or sum(vals)

    agg.append(token_scores)

In [22]:
top_k = 10

for c in range(3):
    print(f"\nTop words for class {id2label[c]}:")

    # positive contribution
    sorted_tokens = sorted(
            [(t, s) for t, s in agg[c].items() if s > 0],
            key=lambda x: x[1],
            reverse=True
        )
    
    for token, score in sorted_tokens[:top_k]:
        print(f"{token}: {score:.4f}")


Top words for class HIGH:
Marseilles.: 0.3132
Alexandria,: 0.1377
Origen: 0.1263
nearly: 0.1041
Ὠριγένης:: 0.0873
Helvetii,: 0.0776
height,: 0.0752
Ptolomaeus: 0.0668
wife: 0.0638
herdsmen.: 0.0613

Top words for class MEDIUM:
senator,: 0.2161
swear,: 0.2103
Sabini: 0.2074
it: 0.1958
Vestini: 0.1945
Rome;: 0.1901
Nomentum: 0.1587
byblus: 0.1454
all.: 0.1229
martyr,: 0.1213

Top words for class LOW:
Marseilles.: 0.2344
Alexandria;: 0.2175
Alexandria,: 0.2048
Boeotia,: 0.1452
Nile,: 0.1080
Stamped: 0.1028
within: 0.0694
Origen:: 0.0558
occupy: 0.0530
them: 0.0497


In [23]:
# negative contribution ?